In [2]:
import arcpy
import os
arcpy.env.addOutputsToMap = False
def get_all_gdbs_from_folder(folder_path, recursive=False):
    """
    获取指定目录下的所有 File GDB 完整路径
    :param folder_path: 目录路径
    :param recursive: 是否递归遍历子目录查找 .gdb
    :return: GDB 完整路径列表
    """
    gdb_list = []
    if not os.path.exists(folder_path):
        print(f"输入目录不存在: '{folder_path}'")
        return gdb_list

    if recursive:
        for root, dirs, _ in os.walk(folder_path):
            for d in dirs:
                if d.lower().endswith(".gdb"):
                    gdb_list.append(os.path.join(root, d))
    else:
        for item in os.listdir(folder_path):
            full_path = os.path.join(folder_path, item)
            if item.lower().endswith(".gdb") and os.path.isdir(full_path):
                gdb_list.append(full_path)

    gdb_list.sort()
    return gdb_list

def merge_feature_classes_from_multiple_gdb(input_gdb_list, output_gdb):
    """
    合并多个FileGDB中同名的要素类（包括要素数据集内的），并保留原有的要素数据集结构，
    最终生成一个新的FileGDB。

    参数:
    - input_gdb_list: 包含所有输入 File GDB 完整路径的列表。
    - output_gdb: 新建输出 File GDB 的完整路径。
    """
    
    # 1. 检查输出 GDB 的父文件夹是否存在
    output_gdb_dir = os.path.dirname(output_gdb)
    if not os.path.exists(output_gdb_dir):
        print(f"输出目录 '{output_gdb_dir}' 不存在，请先创建。")
        return

    # 2. 如果输出 GDB 已存在，则先删除
    if arcpy.Exists(output_gdb):
        print(f"输出 File GDB '{output_gdb}' 已存在，正在删除...")
        arcpy.Delete_management(output_gdb)
    
    # 3. 创建一个新的 File GDB
    arcpy.CreateFileGDB_management(output_gdb_dir, os.path.basename(output_gdb))
    print(f"已创建新的 File GDB: '{output_gdb}'")
    print("-" * 50)

    # 4. 获取所有输入 GDB 中的要素数据集和根目录要素类
    # 修复：改用 arcpy.da.Walk 遍历，完全不依赖 arcpy.env.workspace 状态
    gdb_contents = {}
    for gdb_path in input_gdb_list:
        if not arcpy.Exists(gdb_path):
            print(f"警告: 找不到输入 GDB '{gdb_path}'，已跳过。")
            continue
        
        root_fcs = []
        datasets = {}
        
        for dirpath, dirnames, filenames in arcpy.da.Walk(gdb_path, datatype="FeatureClass"):
            # 规范化路径后比较，避免 Windows 下路径格式不一致
            norm_dirpath = os.path.normcase(os.path.normpath(dirpath))
            norm_gdb_path = os.path.normcase(os.path.normpath(gdb_path))
            if norm_dirpath == norm_gdb_path:
                # 根目录下的要素类
                root_fcs.extend(filenames)
            else:
                # 数据集内的要素类，用 basename 获取数据集名称
                ds_name = os.path.basename(dirpath)
                if ds_name not in datasets:
                    datasets[ds_name] = []
                datasets[ds_name].extend(filenames)
        
        gdb_contents[gdb_path] = {'root_fcs': root_fcs, 'datasets': datasets}

    # 恢复工作空间为初始状态
    arcpy.env.workspace = output_gdb_dir
    
    # 5. 合并根目录下的要素类
    print("开始合并根目录下的要素类...")
    
    # 获取所有 GDB 中根目录要素类的唯一名称
    all_root_fcs = set()
    for gdb in gdb_contents:
        all_root_fcs.update(gdb_contents[gdb]['root_fcs'])
    
    for fc_name in all_root_fcs:
        # 收集所有同名要素类的完整路径
        # 使用 arcpy.Exists 直接判断，避免字典查找漏掉要素类
        input_fcs = [os.path.join(gdb, fc_name) for gdb in gdb_contents if arcpy.Exists(os.path.join(gdb, fc_name))]
        
        if len(input_fcs) > 0:
            output_fc = os.path.join(output_gdb, fc_name)
            
            print(f"  - 正在合并要素类 '{fc_name}'，共找到 {len(input_fcs)} 个源。")
            
            # 合并同名要素类
            try:
                arcpy.Merge_management(input_fcs, output_fc)
                print(f"  - 成功合并到 '{output_fc}'。")
            except Exception as e:
                print(f"  - 合并 '{fc_name}' 时出错: {e}")
    
    print("-" * 50)
    
    # 6. 合并要素数据集及其中的要素类
    print("开始合并要素数据集及其中的要素类...")

    # 获取所有 GDB 中要素数据集的唯一名称
    all_datasets = set()
    for gdb in gdb_contents:
        all_datasets.update(gdb_contents[gdb]['datasets'].keys())
    
    for dataset_name in all_datasets:
        # 在输出 GDB 中创建同名的要素数据集
        output_dataset = os.path.join(output_gdb, dataset_name)
        
        # 获取第一个包含该数据集的 GDB，以获取空间参考
        first_gdb_with_dataset = next((gdb for gdb, content in gdb_contents.items() if dataset_name in content['datasets']), None)
        if first_gdb_with_dataset:
            # 获取该数据集的空间参考
            first_dataset_path = os.path.join(first_gdb_with_dataset, dataset_name)
            try:
                spatial_ref = arcpy.Describe(first_dataset_path).spatialReference
            except AttributeError:
                # 回退：从数据集内第一个要素类获取空间参考
                fcs_in_ds = gdb_contents[first_gdb_with_dataset]['datasets'].get(dataset_name, [])
                if fcs_in_ds:
                    first_fc_path = os.path.join(first_dataset_path, fcs_in_ds[0])
                    spatial_ref = arcpy.Describe(first_fc_path).spatialReference
                else:
                    spatial_ref = arcpy.SpatialReference(4490)  # 默认 CGCS2000
            
            arcpy.CreateFeatureDataset_management(output_gdb, dataset_name, spatial_ref)
            print(f"  - 已在输出 GDB 中创建要素数据集 '{dataset_name}'。")

            # 获取所有 GDB 中该要素数据集内的要素类名称
            all_dataset_fcs = set()
            for gdb in gdb_contents:
                if dataset_name in gdb_contents[gdb]['datasets']:
                    all_dataset_fcs.update(gdb_contents[gdb]['datasets'][dataset_name])

            # 遍历并合并每个要素数据集内的要素类
            for fc_name in all_dataset_fcs:
                input_fcs = []
                for gdb in gdb_contents:
                    # 使用 arcpy.Exists 直接判断，绕过字典查找
                    fc_full_path = os.path.join(gdb, dataset_name, fc_name)
                    if arcpy.Exists(fc_full_path):
                        input_fcs.append(fc_full_path)
                
                if len(input_fcs) > 0:
                    output_fc = os.path.join(output_dataset, fc_name)
                    
                    print(f"  - 正在合并要素数据集 '{dataset_name}' 下的要素类 '{fc_name}'，共找到 {len(input_fcs)} 个源。")
                    
                    try:
                        arcpy.Merge_management(input_fcs, output_fc)
                        print(f"  - 成功合并到 '{output_fc}'。")
                    except Exception as e:
                        print(f"  - 合并 '{fc_name}' 时出错: {e}")
    
    print("-" * 50)
    print("所有要素类合并完成。")


if __name__ == '__main__':
    # ----------------------------------------------------
    # 用户需要修改以下参数
    # ----------------------------------------------------

    # 1. 指定包含待合并 GDB 的目录路径（不再需要手动逐个列出 GDB）
    input_folder = r"D:\Pro Projects\CAD处理\CAD\2026年22个区县数据库合并\需合库数据\需合库数据"
    
    # 是否递归查找子目录中的 .gdb (如果 GDB 都在 input_folder 根目录下，设为 False；如果分散在各子文件夹，设为 True)
    recursive_search = False

    # 2. 指定新创建的输出 File GDB 的完整路径
    output_gdb_path = r"D:\Pro Projects\CAD处理\CAD\2026年22个区县数据库合并\需合库数据\CDData20261012.gdb"

    # ----------------------------------------------------
    # 自动检索待合并 GDB 列表
    # ----------------------------------------------------
    input_gdb_paths = get_all_gdbs_from_folder(input_folder, recursive=recursive_search)

    # 安全检查：若输出 GDB 位于输入目录下，自动排除自身以避免重复读入或冲突
    norm_output = os.path.normcase(os.path.normpath(output_gdb_path))
    input_gdb_paths = [p for p in input_gdb_paths if os.path.normcase(os.path.normpath(p)) != norm_output]

    print(f"在目录 '{input_folder}' 下共检索到 {len(input_gdb_paths)} 个待合并 GDB 数据库:")
    for p in input_gdb_paths:
        print(f"  - {os.path.basename(p)}")
    print("-" * 50)

    # ----------------------------------------------------
    # 调用函数，开始执行脚本
    # ----------------------------------------------------
    if input_gdb_paths:
        merge_feature_classes_from_multiple_gdb(input_gdb_paths, output_gdb_path)
    else:
        print("警告: 未在指定目录下找到任何待合并的 GDB 数据库，已终止执行。")


在目录 'D:\Pro Projects\CAD处理\CAD\2026年22个区县数据库合并\需合库数据\需合库数据' 下共检索到 22 个待合并 GDB 数据库:
  - 510104锦江区_CSJC2026.gdb
  - 510105青羊区_CSJC2026.gdb
  - 510106金牛区_CSJC2026.gdb
  - 510107武侯区_CSJC2026.gdb
  - 510108成华区_CSJC2026.gdb
  - 510109高新区_CSJC2026.gdb
  - 510110天府新区成都直管区_CSJC2026.gdb
  - 510112龙泉驿区_CSJC2026.gdb
  - 510113青白江区_CSJC2026.gdb
  - 510114新都区_CSJC2026.gdb
  - 510115温江区_CSJC2026.gdb
  - 510116双流区_CSJC2026.gdb
  - 510117郫都区_CSJC2026.gdb
  - 510118新津区_CSJC2026.gdb
  - 510121金堂县_CSJC2026.gdb
  - 510129大邑县_CSJC2026.gdb
  - 510131蒲江县_CSJC2026.gdb
  - 510181都江堰市_CSJC2026.gdb
  - 510182彭州市_CSJC2026.gdb
  - 510183邛崃市_CSJC2026.gdb
  - 510184崇州市_CSJC2026.gdb
  - 510185简阳市_CSJC2026.gdb
--------------------------------------------------
已创建新的 File GDB: 'D:\Pro Projects\CAD处理\CAD\2026年22个区县数据库合并\需合库数据\CDData20261012.gdb'
--------------------------------------------------
开始合并根目录下的要素类...
  - 正在合并要素类 'KFQ'，共找到 22 个源。
  - 成功合并到 'D:\Pro Projects\CAD处理\CAD\2026年22个区县数据库合并\需合库数据\CDData20261012.gdb\KFQ'